# Peer Comparison Engine

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

from analytics.db import run_query

fact_analysis = run_query("""
SELECT *
FROM fact_analysis
""")

fact_profit_loss = run_query("""
SELECT *
FROM fact_profit_loss
""")

fact_balance_sheet = run_query("""
SELECT *
FROM fact_balance_sheet
""")

def extract_percentage(value):

    if pd.isnull(value):
        return None

    try:
        return float(
            str(value)
            .split(":")[-1]
            .replace("%", "")
            .strip()
        )

    except:
        return None

# Feature Engineering

In [2]:
fact_analysis["sales_growth_pct"] = fact_analysis[
    "compounded_sales_growth"
].apply(extract_percentage)

fact_analysis["profit_growth_pct"] = fact_analysis[
    "compounded_profit_growth"
].apply(extract_percentage)

fact_analysis["roe_pct"] = fact_analysis[
    "roe"
].apply(extract_percentage)

In [5]:
latest_profit = fact_profit_loss.sort_values(
    "sorting_order",
    ascending=False
).groupby("company_id").first().reset_index()

latest_balance = fact_balance_sheet.sort_values(
    "sorting_order",
    ascending=False
).groupby("company_id").first().reset_index()

latest_analysis = fact_analysis.groupby(
    "company_id"
).first().reset_index()

peer_df = latest_profit.merge(
    latest_balance,
    on="company_id",
    how="left"
)

peer_df = peer_df.merge(
    latest_analysis[
        [
            "company_id",
            "sales_growth_pct",
            "profit_growth_pct",
            "roe_pct"
        ]
    ],
    on="company_id",
    how="left"
)

peer_features = peer_df[
    [
        "opm_percentage",
        "net_profit_margin_pct",
        "roe_pct",
        "sales_growth_pct",
        "profit_growth_pct",
        "debt_to_equity"
    ]
].fillna(0)

In [6]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(
    peer_features
)

In [8]:
similarity_matrix = cosine_similarity(
    scaled_features
)
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=peer_df["company_id"],
    columns=peer_df["company_id"]
)

In [9]:
peer_mapping = {}

for company in similarity_df.index:

    similarities = similarity_df.loc[company]

    top_peers = similarities.sort_values(
        ascending=False
    )[1:6]

    peer_mapping[company] = list(top_peers.index)

In [10]:
peer_mapping["TCS"]

['INFY', 'WIPRO', 'SBILIFE', 'HDFCBANK', 'BAJAJHLDNG']

In [11]:
peer_mapping["INFY"]

['TCS', 'WIPRO', 'SBILIFE', 'HDFCBANK', 'BAJAJHLDNG']

In [12]:
peer_mapping["HDFCBANK"]

['INFY', 'SBILIFE', 'TCS', 'WIPRO', 'ICICIBANK']

In [13]:
peer_output = pd.DataFrame(
    [
        {
            "company_id": company,
            "peers": ", ".join(peers)
        }

        for company, peers in peer_mapping.items()
    ]
)

peer_output.to_csv(
    "../exports/peer_mapping.csv",
    index=False
)